# R1-C01 — Generación anual autorregresiva RAW 2023 · V01

**Estado:** listo para auditoría del código; **ejecución anual bloqueada por defecto**. El investigador debe autorizar explícitamente la ejecución solo después de aprobar el código y los controles.

Este notebook reproduce el motor RAW usado y verificado en el *preflight* de cuatro bloques. No entrena, reajusta escaladores, usa observaciones de 2023 como entradas radiativas, ni aplica POST_CANDIDATE / FINAL. Datos históricos permitidos únicamente en semilla de diciembre de 2022. Las entradas futuras proceden del paquete de preparación auditado.

**Entradas en `/content/`:** modelo `.keras`, metadatos `.json`, los dos `.joblib`, `R1_C01_PREP_2023.zip`, `R1_C01_FIRSTBLOCK_2023_V01.zip` y `R1_C01_2023_PREFLIGHT_4BLOCKS_V01.zip`.

**Destino documental:** `REV_R1_C01_ANNUAL_BACKTEST/03_ROLLOUT_2023`. No sobrescribir los artefactos canónicos.

**Etapas:** 1–5 cargar/validar, 6 comparar cuatro bloques, 7 autorizar (NO por defecto), 8 generar 730 bloques si está autorizado, 9 auditar/exportar, 10 empaquetar evidencia.


In [ ]:
# 1 — Configuración e integridad criptográfica
from pathlib import Path
import json, io, hashlib, zipfile, sys, platform
import numpy as np, pandas as pd, joblib

YEAR=2023; L=72; H=24; EXPECTED_STEPS=17520; EXPECTED_BLOCKS=730
ROOT=Path('/content'); OUTPUT=ROOT/'R1_C01_ROLLOUT_RAW_2023_V01'
OUTPUT.mkdir(parents=True,exist_ok=True)
PREP=ROOT/'R1_C01_PREP_2023.zip'
FIRST=ROOT/'R1_C01_FIRSTBLOCK_2023_V01.zip'
MODEL='PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras'
EXPECTED_HASH={
 MODEL:'5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e',
 'model_metadata_v24_v1.json':'3b35e72dc8f2a8d9d9a083e7f70be1fadf2e6429363777d673adac7567f397be',
 'scaler_hist_v24_v1.joblib':'a7189921396242f683bd763b4252667e5a7a978c652b039abcbbb1551e586129',
 'scaler_fut_v24_v1.joblib':'b4917f7df32a5e4ade2148cbb9d42e1af7a4a6696f1a0a2b5ec6382d44421e9e',
}
def sha_bytes(data):return hashlib.sha256(data).hexdigest()
def sha_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for buf in iter(lambda:f.read(1024*1024),b''):h.update(buf)
 return h.hexdigest()
def locate(name):
 candidates=[ROOT/name,ROOT/('R1_C01_PREP_'+str(YEAR))/name]
 found=[p for p in candidates if p.is_file()]
 if not found:found=sorted(p for p in ROOT.rglob(name) if p.is_file())
 if len(found)!=1:raise RuntimeError(f'Ubicación ambigua o ausente: {name}: {found}')
 return found[0]
def zipread(zpath,name):
 if not zpath.is_file():raise FileNotFoundError(zpath)
 with zipfile.ZipFile(zpath) as z:return z.read(name)
def prep_numpy(name):return np.load(io.BytesIO(zipread(PREP,name)),allow_pickle=False)
def prep_csv(name):return pd.read_csv(io.BytesIO(zipread(PREP,name)),parse_dates=['timestamp'])
paths={name:locate(name) for name in EXPECTED_HASH}
for name,path in paths.items():
 actual=sha_file(path);assert actual==EXPECTED_HASH[name],f'Hash canónico no coincide: {name}\n{actual}'
assert PREP.is_file() and FIRST.is_file(),'Faltan ZIP de preparación o primer bloque'
pmanifest=json.loads(zipread(PREP,'R1_C01_2023_Preparation_Manifest_V01.json'))
for name,h in pmanifest['output_sha256'].items():
 assert sha_bytes(zipread(PREP,name))==h, f'Hash preparación: {name}'
fmanifest=json.loads(zipread(FIRST,'R1_C01_2023_FirstBlock_Manifest_V01.json'))
for name,h in fmanifest['outputs_sha256'].items():
 assert sha_bytes(zipread(FIRST,name))==h, f'Hash primer bloque: {name}'
assert fmanifest['model_sha256']==EXPECTED_HASH[MODEL]
print('PASS: modelo, metadatos, scalers y ZIP de preparación y primer bloque')

In [ ]:
# 2 — Contrato, matrices, separación temporal y escaladores
meta=json.loads(paths['model_metadata_v24_v1.json'].read_text(encoding='utf-8'))
HIST_COLS=list(meta['hist_cols']);FUT_COLS=list(meta['future_cols'])
assert len(HIST_COLS)==34 and len(FUT_COLS)==17
assert (int(meta['L']),int(meta['H']))==(L,H)
assert pmanifest['features_hist']==HIST_COLS and pmanifest['features_future']==FUT_COLS
seed=prep_csv('R1_C01_2023_Seed_RawFeatures_V01.csv')
future=prep_csv('R1_C01_2023_Future_RawFeatures_V01.csv')
X_seed=prep_numpy('R1_C01_2023_X_hist_seed_V01.npy')
X_future=prep_numpy('R1_C01_2023_X_future_V01.npy')
assert seed.shape[0]==72 and future.shape[0]==EXPECTED_STEPS
assert X_seed.shape==(72,34) and X_future.shape==(EXPECTED_STEPS,17)
assert X_seed.dtype==X_future.dtype==np.float32
assert np.isfinite(X_seed).all() and np.isfinite(X_future).all()
assert seed.timestamp.iloc[-1]==pd.Timestamp('2022-12-31 23:30:00')
assert (seed.timestamp<pd.Timestamp('2023-01-01')).all()
assert future.timestamp.iloc[0]==pd.Timestamp('2023-01-01 00:00')
assert future.timestamp.iloc[-1]==pd.Timestamp('2023-12-31 23:30')
assert future.timestamp.is_unique
assert (future.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
assert set(future.timestamp.dt.year)=={2023}
assert 'GHI' not in FUT_COLS and 'DNI' not in FUT_COLS and 'DHI' not in FUT_COLS
scaler_h=joblib.load(paths['scaler_hist_v24_v1.joblib']);scaler_f=joblib.load(paths['scaler_fut_v24_v1.joblib'])
assert scaler_h.n_features_in_==34 and scaler_f.n_features_in_==14
if hasattr(scaler_h,'feature_names_in_'):assert list(scaler_h.feature_names_in_)==HIST_COLS
if hasattr(scaler_f,'feature_names_in_'):assert list(scaler_f.feature_names_in_)==FUT_COLS[:14]
np.testing.assert_allclose(scaler_h.transform(seed[HIST_COLS].astype('float32')).astype('float32'),X_seed,atol=2e-5,rtol=2e-6)
X_f_recalc=np.column_stack([scaler_f.transform(future[FUT_COLS[:14]].astype('float32')).astype('float32'),future[FUT_COLS[14:]].to_numpy(dtype='float32')]).astype('float32')
np.testing.assert_allclose(X_f_recalc,X_future,atol=2e-5,rtol=2e-6)
assert (future['Clearsky DNI']>=0).all() and (future['Clearsky DHI']>=0).all()
print('PASS: corte temporal, semilla, 730 bloques, contrato 34/17, escala y entradas finitas')

In [ ]:
# 3 — Carga funcional, sin compilación ni entrenamiento
import tensorflow as tf
class Expm1Layer(tf.keras.layers.Layer):
 def call(self,inputs):return tf.math.expm1(inputs)
 def get_config(self):return super().get_config()
model=tf.keras.models.load_model(paths[MODEL],custom_objects={'Expm1Layer':Expm1Layer},compile=False)
assert model.count_params()==110146
assert [tuple(x.shape) for x in model.inputs]==[(None,72,34),(None,24,17)]
assert tuple(model.output.shape)==(None,24,2)
assert str(meta['target_direct'])=='kDNI' and str(meta['target_diffuse'])=='qDHI=log1p(kDHI)'
print('PASS Keras:',tf.__version__,model.name,model.count_params())

In [ ]:
# 4 — Reconstrucción de características: misma implementación usada en el primer bloque aprobado
class Config:
    thr_cs=20.0; kdni_max=1.2; kdhi_max=8.0; qdhi_max=float(np.log1p(8.0))
cfg2050=Config()
COL_GHI='GHI';COL_DNI='DNI';COL_DHI='DHI'
COL_CSGHI='Clearsky GHI';COL_CSDNI='Clearsky DNI';COL_CSDHI='Clearsky DHI'

def build_historical_features_v24v1(df):
    out = df.copy().sort_values("timestamp").reset_index(drop=True)
    eps_local = 1e-6

    # --------------------------------------------------------
    # Índices radiativos
    # --------------------------------------------------------
    csghi = out[COL_CSGHI].to_numpy(dtype=float)
    csdni = out[COL_CSDNI].to_numpy(dtype=float)
    csdhi = out[COL_CSDHI].to_numpy(dtype=float)

    ghi = out[COL_GHI].to_numpy(dtype=float)
    dni = out[COL_DNI].to_numpy(dtype=float)
    dhi = out[COL_DHI].to_numpy(dtype=float)

    kg = np.zeros(len(out), dtype=float)
    kn = np.zeros(len(out), dtype=float)
    kd = np.zeros(len(out), dtype=float)

    m = csghi > cfg2050.thr_cs
    kg[m] = ghi[m] / np.maximum(csghi[m], eps_local)

    m = csdni > cfg2050.thr_cs
    kn[m] = dni[m] / np.maximum(csdni[m], eps_local)

    m = csdhi > cfg2050.thr_cs
    kd[m] = dhi[m] / np.maximum(csdhi[m], eps_local)

    out["kGHI"] = np.clip(kg, 0.0, 2.0)
    out["kDNI"] = np.clip(kn, 0.0, cfg2050.kdni_max)
    out["kDHI"] = np.clip(kd, 0.0, cfg2050.kdhi_max)
    out["qDHI"] = np.log1p(out["kDHI"])

    # --------------------------------------------------------
    # Ratios radiativos
    # --------------------------------------------------------
    ghi_safe = np.maximum(ghi, eps_local)

    out["diffuse_ratio"] = np.clip(
        dhi / ghi_safe, 0.0, 1.2
    )

    out["beam_ratio"] = np.clip(
        dni * out["cosz_plus"].to_numpy(dtype=float) / ghi_safe,
        0.0, 1.2
    )

    # --------------------------------------------------------
    # Diferencias
    # --------------------------------------------------------
    out["dGHI"] = out[COL_GHI].diff()
    out["dDNI"] = out[COL_DNI].diff()
    out["dDHI"] = out[COL_DHI].diff()

    # --------------------------------------------------------
    # Irradiancia: lag + rolling
    # --------------------------------------------------------
    out["GHI_lag1"] = out[COL_GHI].shift(1)

    out["DNI_lag1"] = out[COL_DNI].shift(1)
    out["DNI_roll3"] = out[COL_DNI].rolling(3).mean()

    out["DHI_lag1"] = out[COL_DHI].shift(1)
    out["DHI_roll3"] = out[COL_DHI].rolling(3).mean()

    # --------------------------------------------------------
    # kGHI
    # --------------------------------------------------------
    out["kGHI_lag1"] = out["kGHI"].shift(1)
    out["kGHI_roll3"] = out["kGHI"].rolling(3).mean()

    # --------------------------------------------------------
    # kDNI
    # --------------------------------------------------------
    out["kDNI_lag1"] = out["kDNI"].shift(1)
    out["kDNI_roll3"] = out["kDNI"].rolling(3).mean()

    # --------------------------------------------------------
    # qDHI — representación difusa V2.4
    # --------------------------------------------------------
    out["qDHI_lag1"] = out["qDHI"].shift(1)
    out["qDHI_roll3"] = out["qDHI"].rolling(3).mean()

    # --------------------------------------------------------
    # Ratios
    # --------------------------------------------------------
    out["diffuse_ratio_lag1"] = out["diffuse_ratio"].shift(1)
    out["diffuse_ratio_roll3"] = out["diffuse_ratio"].rolling(3).mean()

    out["beam_ratio_lag1"] = out["beam_ratio"].shift(1)
    out["beam_ratio_roll3"] = out["beam_ratio"].rolling(3).mean()

    return out

assert HIST_COLS==list(meta["hist_cols"])
print("PASS: función canónica disponible")

In [ ]:
# 5 — Motor RAW reutilizable; sin intervención POST/FINAL
STATE_COLS=['timestamp','GHI','DNI','DHI','cosz_plus','Clearsky GHI','Clearsky DNI','Clearsky DHI','Temperature','Dew Point','Relative Humidity','Pressure','Aerosol Optical Depth','Precipitable Water','Wind Speed','sin_wdir','cos_wdir']
assert set(STATE_COLS).issubset(seed.columns) and set(STATE_COLS).issubset(future.columns.union(pd.Index(['GHI','DNI','DHI'])))
KDNI_MAX=float(meta['kdni_max']);QDHI_MAX=float(meta['qdhi_max']);KDHI_MAX=float(meta['kdhi_max'])

def run_raw(n_blocks):
 assert isinstance(n_blocks,int) and 1<=n_blocks<=EXPECTED_BLOCKS
 state_base=seed[STATE_COLS].copy().reset_index(drop=True)
 state_scaled=X_seed.copy(); all_blocks=[]; audit=[]
 for block_id in range(n_blocks):
  i0=block_id*H;i1=i0+H
  assert np.isfinite(state_scaled).all()
  xb=X_future[i0:i1][np.newaxis,...].astype(np.float32)
  yh=state_scaled[np.newaxis,...].astype(np.float32)
  assert yh.shape==(1,72,34) and xb.shape==(1,24,17)
  pred=np.asarray(model([yh,xb],training=False))[0]
  assert pred.shape==(24,2) and np.isfinite(pred).all(),f'Predicción inválida bloque {block_id+1}'
  assert ((pred[:,0]>=-1e-6)&(pred[:,0]<=KDNI_MAX+1e-6)).all()
  assert ((pred[:,1]>=-1e-6)&(pred[:,1]<=QDHI_MAX+1e-6)).all()
  kn=np.clip(pred[:,0].astype(float),0,KDNI_MAX)
  qd=np.clip(pred[:,1].astype(float),0,QDHI_MAX)
  kd=np.clip(np.expm1(qd),0,KDHI_MAX)
  fut=future.iloc[i0:i1].copy()
  cosz=fut['cosz_plus'].to_numpy(dtype=float)
  csdni=fut['Clearsky DNI'].to_numpy(dtype=float)
  csdhi=fut['Clearsky DHI'].to_numpy(dtype=float)
  dni=kn*csdni;dhi=kd*csdhi;ghi=dni*cosz+dhi
  night=cosz<=0
  dni[night]=0;dhi[night]=0;ghi[night]=0
  assert all(np.isfinite(x).all() for x in (ghi,dni,dhi))
  assert min(ghi.min(),dni.min(),dhi.min())>=0
  closure=ghi-(dni*cosz+dhi)
  cerrmax=float(np.abs(closure).max())
  assert cerrmax<1e-6
  assert np.count_nonzero(ghi[night])+np.count_nonzero(dni[night])+np.count_nonzero(dhi[night])==0
  generated=pd.DataFrame({'timestamp':fut.timestamp.to_numpy(),'block_id':block_id,'horizon_step':np.arange(1,H+1),'cosz_plus':cosz,'Clearsky GHI':fut['Clearsky GHI'].to_numpy(dtype=float),'Clearsky DNI':csdni,'Clearsky DHI':csdhi,'kDNI_raw':kn,'qDHI_raw':qd,'kDHI_raw':kd,'DNI_raw':dni,'DHI_raw':dhi,'GHI_raw':ghi})
  all_blocks.append(generated)
  synthetic=fut[STATE_COLS[0:1]+STATE_COLS[4:]].copy()
  synthetic['GHI']=ghi;synthetic['DNI']=dni;synthetic['DHI']=dhi
  synthetic=synthetic[STATE_COLS]
  combined=pd.concat([state_base,synthetic],ignore_index=True)
  new_features=build_historical_features_v24v1(combined).iloc[-L:].copy().reset_index(drop=True)
  assert np.isfinite(new_features[HIST_COLS].to_numpy(dtype=float)).all()
  state_scaled=scaler_h.transform(new_features[HIST_COLS].astype('float32')).astype('float32')
  state_base=combined.iloc[-L:].copy().reset_index(drop=True)
  assert state_scaled.shape==(72,34) and np.isfinite(state_scaled).all()
  observed_in_context=max(0,72-(block_id+1)*24)
  # Contexto inicial: 72 observaciones. Tras los bloques 1,2,3 quedan 48,24,0.
  assert (state_base.timestamp<pd.Timestamp('2023-01-01')).sum()==observed_in_context
  audit.append({'block_number':block_id+1,'start':str(generated.timestamp.iloc[0]),'end':str(generated.timestamp.iloc[-1]),'historical_rows_after_block':observed_in_context,'synthetic_rows_after_block':72-observed_in_context,'GHI_energy_kWh_m2':float(ghi.sum()*0.0005),'DNI_energy_kWh_m2':float(dni.sum()*0.0005),'DHI_energy_kWh_m2':float(dhi.sum()*0.0005),'GHI_max_Wm2':float(ghi.max()),'DNI_max_Wm2':float(dni.max()),'DHI_max_Wm2':float(dhi.max()),'closure_max_Wm2':cerrmax,'night_leakage':0,'kdni_max':float(kn.max()),'kdhi_max':float(kd.max())})
  if (block_id+1)%50==0:print('Procesados',block_id+1,'/',n_blocks)
 result=pd.concat(all_blocks,ignore_index=True)
 assert result.timestamp.is_unique and len(result)==n_blocks*H
 assert (result.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
 return result,pd.DataFrame(audit),state_base,state_scaled
print('PASS: motor definido; sin rollout anual ejecutado')

In [ ]:
# 6 — Regresión preflight de 4 bloques: equivalencia independiente contra ZIP ya auditado
PREFLIGHT = ROOT / 'R1_C01_2023_PREFLIGHT_4BLOCKS_V01.zip'
assert PREFLIGHT.is_file(), 'Falta ZIP del preflight aprobado; detener.'
pf_manifest=json.loads(zipread(PREFLIGHT,'R1_C01_2023_Preflight_4Blocks_Manifest_V01.json'))
assert pf_manifest['status']=='PREFLIGHT_4_BLOCKS_ONLY'
assert pf_manifest['blocks_executed']==4
assert pf_manifest['input_model_sha256']==EXPECTED_HASH[MODEL]
for name,expected in pf_manifest['files_sha256'].items():
    assert sha_bytes(zipread(PREFLIGHT,name))==expected, f'ZIP preflight no válido: {name}'
short_raw,short_audit,_,short_scaled=run_raw(4)
reference=pd.read_csv(io.BytesIO(zipread(FIRST,'R1_C01_2023_FirstBlock_RAW_V01.csv')),parse_dates=['timestamp'])
assert reference.timestamp.reset_index(drop=True).equals(short_raw.timestamp.iloc[:24].reset_index(drop=True))
for reference_col,new_col in [('GHI_pred','GHI_raw'),('DNI_pred','DNI_raw'),('DHI_pred','DHI_raw'),('kDNI_pred','kDNI_raw'),('qDHI_pred','qDHI_raw'),('kDHI_pred','kDHI_raw')]:
    np.testing.assert_allclose(reference[reference_col].to_numpy(),short_raw[new_col].iloc[:24].to_numpy(),atol=1e-5,rtol=1e-6)
pf_raw=pd.read_csv(io.BytesIO(zipread(PREFLIGHT,'R1_C01_2023_RAW_4Blocks_Preflight_V01.csv')),parse_dates=['timestamp'])
pf_audit=pd.read_csv(io.BytesIO(zipread(PREFLIGHT,'R1_C01_2023_Audit_4Blocks_Preflight_V01.csv')))
assert pf_raw.timestamp.equals(short_raw.timestamp)
assert list(short_audit.historical_rows_after_block)==[48,24,0,0]
assert pf_audit.shape[0]==4
for col in ('GHI_raw','DNI_raw','DHI_raw','kDNI_raw','qDHI_raw','kDHI_raw'):
    np.testing.assert_allclose(pf_raw[col],short_raw[col],atol=1e-5,rtol=1e-6)
assert np.isfinite(short_scaled).all()
print('PASS REGRESIÓN: 4 bloques equivalentes al preflight aprobado, sin anomalías.')


## 7. Autorización explícita del rollout anual

**Por defecto el rollout está deshabilitado.** No cambies el valor siguiente hasta que se revise y autorice esta V01. Para ejecutar posteriormente, sustituye exclusivamente `NO_AUTORIZADO` por `AUTORIZO_R1_C01_RAW_2023_730_BLOQUES` en la celda 7 y vuelve a ejecutar a partir de ella. La celda 8 solo generará el año si existe una autorización exacta y el preflight superó las pruebas.


In [ ]:
# 7 — CONTROL DE AUTORIZACIÓN: dejar deshabilitado durante auditoría
AUTORIZACION_ANUAL = 'NO_AUTORIZADO'
TOKEN_ANUAL = 'AUTORIZO_R1_C01_RAW_2023_730_BLOQUES'
AUTHORIZED = AUTORIZACION_ANUAL == TOKEN_ANUAL
print('Autorización anual activa:', AUTHORIZED)
if not AUTHORIZED:
    print('BLOQUEO ACTIVO: no se ejecutarán los 730 bloques. Notebook listo para revisión.')


In [ ]:
# 8 — GENERACIÓN 2023 RAW: únicamente si la autorización exacta fue activada
ANNUAL_COMPLETED = False
annual_raw = annual_audit = annual_state = annual_scaled = None
if not AUTHORIZED:
    print('SKIP: rollout anual no autorizado.')
else:
    assert 'short_raw' in globals() and len(short_raw)==96, 'Ejecuta antes el preflight.'
    assert EXPECTED_BLOCKS==730 and EXPECTED_STEPS==17520
    assert not list(OUTPUT.glob('R1_C01_2023_Annual_RAW_V01.csv')), 'Ya existe salida anual: no sobrescribir.'
    annual_raw,annual_audit,annual_state,annual_scaled=run_raw(EXPECTED_BLOCKS)
    assert len(annual_raw)==EXPECTED_STEPS and len(annual_audit)==EXPECTED_BLOCKS
    assert annual_raw.timestamp.iloc[0]==pd.Timestamp('2023-01-01 00:00:00')
    assert annual_raw.timestamp.iloc[-1]==pd.Timestamp('2023-12-31 23:30:00')
    assert annual_raw.timestamp.is_unique and (annual_raw.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
    assert (annual_raw[['GHI_raw','DNI_raw','DHI_raw']]>=0).all().all()
    assert np.isfinite(annual_raw[['GHI_raw','DNI_raw','DHI_raw']].to_numpy()).all()
    assert annual_audit['night_leakage'].eq(0).all()
    assert annual_audit['closure_max_Wm2'].max()<1e-6
    assert list(annual_audit['historical_rows_after_block'].iloc[:4])==[48,24,0,0]
    assert annual_audit['historical_rows_after_block'].iloc[3:].eq(0).all()
    assert np.isfinite(annual_scaled).all()
    for col in ('GHI_raw','DNI_raw','DHI_raw','kDNI_raw','qDHI_raw','kDHI_raw'):
        np.testing.assert_allclose(annual_raw[col].iloc[:96],short_raw[col],atol=1e-5,rtol=1e-6)
    ANNUAL_COMPLETED=True
    print('PASS ROLLOUT: 730 bloques, 17 520 instantes, RAW sin posproceso.')


In [ ]:
# 9 — EXPORTACIÓN SEGURA: RAW y auditoría por bloque; sin POST ni FINAL
annual_files={}
if not ANNUAL_COMPLETED:
    print('SKIP exportación: no hubo ejecución anual autorizada.')
else:
    names=[('R1_C01_2023_Annual_RAW_V01.csv',annual_raw),('R1_C01_2023_Annual_RAW_BlockAudit_V01.csv',annual_audit)]
    for name,frame in names:
        path=OUTPUT/name
        assert not path.exists(), f'Ya existe: {path}; nunca sobrescribir.'
        frame.to_csv(path,index=False)
        annual_files[name]=sha_file(path)
    manifest={
        'project':'R1-C01 retrospective annual autoregressive validation',
        'year':2023,'status':'RAW_ANNUAL_EXECUTED_UNEVALUATED',
        'model':'PI-LSTM V2.4 V1 RAW','annual_rollout_executed':True,
        'postprocess_applied':False,'number_of_blocks':730,'number_of_rows':17520,
        'initial_observation_end':'2022-12-31 23:30:00',
        'reference_climatology_years':'1998-2022',
        'input_model_sha256':EXPECTED_HASH[MODEL],
        'input_metadata_sha256':EXPECTED_HASH['model_metadata_v24_v1.json'],
        'input_hist_scaler_sha256':EXPECTED_HASH['scaler_hist_v24_v1.joblib'],
        'input_fut_scaler_sha256':EXPECTED_HASH['scaler_fut_v24_v1.joblib'],
        'input_preparation_zip_sha256':sha_file(PREP),
        'input_firstblock_zip_sha256':sha_file(FIRST),
        'input_preflight_zip_sha256':sha_file(PREFLIGHT),
        'outputs_sha256':annual_files,
        'python':platform.python_version(),'tensorflow':tf.__version__,
        'numpy':np.__version__,'pandas':pd.__version__,
        'scikit_learn':__import__('sklearn').__version__,
        'closure_max_Wm2':float(annual_audit['closure_max_Wm2'].max()),
        'total_energy_kWh_m2':{k:float(annual_raw[k].sum()*0.0005) for k in ('GHI_raw','DNI_raw','DHI_raw')},
        'note':'Numerical audit only; independent statistical validation against NSRDB not yet performed.'
    }
    manifest_path=OUTPUT/'R1_C01_2023_Annual_RAW_Manifest_V01.json'
    assert not manifest_path.exists(), 'Manifiesto existente: evitar sobrescritura.'
    manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    print('ARCHIVOS ANUALES:',list(annual_files),'y',manifest_path.name)
    print('GHI/DNI/DHI anual kWh/m²:',manifest['total_energy_kWh_m2'])


In [ ]:
# 10 — EMPAQUETADO DE EVIDENCIA (solo con rollout completado)
if not ANNUAL_COMPLETED:
    print('SKIP ZIP: no hubo ejecución anual autorizada.')
else:
    final_zip=ROOT/'R1_C01_2023_ANNUAL_RAW_V01.zip'
    assert not final_zip.exists(), 'ZIP existente: evita sobrescribir una ejecución previa.'
    with zipfile.ZipFile(final_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for name in list(annual_files)+['R1_C01_2023_Annual_RAW_Manifest_V01.json']:
            z.write(OUTPUT/name,arcname=name)
    print('ZIP:',final_zip,'SHA256:',sha_file(final_zip))
    print('Conserva el ZIP y los CSV en 03_ROLLOUT_2023 y envíalos para auditoría científica.')


## Criterios de revisión antes de autorizar

- SHA-256 del modelo, escaladores y metadatos; hashes de ZIP de preparación, primer bloque y preflight.
- Semilla observada únicamente hasta `2022-12-31 23:30`; no se cargan observaciones radiativas 2023.
- Contrato de **34** entradas históricas y **17** futuras, sin reescalado de clear-sky; actualización a partir de predicciones RAW.
- Regresión de cuatro bloques contra evidencia aprobada; contexto histórico observado se extingue después del bloque 3.
- Cierre radiativo, finitud, no negatividad, cero nocturno; continuidad temporal y auditabilidad por bloque.
- En modo autorizado, exportación de CSV RAW, diagnósticos de 730 bloques y manifiesto. **No hay POST_CANDIDATE ni FINAL.**
- Nunca interpretar PASS numérico como representatividad energética o distribucional anual; esa evaluación es una fase independiente.

**Revisión estática no sustituye ejecución en Colab.** La autorización explícita debe darse después de auditar este notebook. No cambiar los archivos congelados ni las funciones de reconstrucción aprobadas.
